This funtion attempts to test the piecewiseInterpRotMat function in gen_utilities.py.
This function interpolates each term of the rotation matrices of an element.

It also test the orthogonal nature of rotation matrix.

In [1]:
import sys
import numpy as np
from pathlib import Path

# Starting from Model/testing/, go up one level to Model/
project_root = Path.cwd().parent
sys.path.append(str(project_root))

from gen.gen_utilities import interpLagGLQ, rotTensor, rotVector

In [2]:
def piecewiseInterpRotMat(Sarray, rotMatArray, NNPEL):
    '''
    This function interpolates each term of the given rotation matrices/tensors of an element.
    It vectorises the rotation matrix at each node and creates a numpy array of vectorised rotation matrix of size (NNPEL, 9).
    Subsequently, it uses Sarray to interpolate and calculate vectorised rotation matrix at Gauss point of size numpy.ndarray(9,1).
    This vectorised rotation matrix of a Gauss point is then devectorised to create a numpy array of size (3,3).
    
    Sarray: numpy array of shape (NNPEL) containing interpolation function values at a particular Gauss point.
    rotMatArray: numpy array of shape (NNPEL, 3, 3) containing rotation tensors at each node of the element.
    NNPEL: number of nodes per element. It could also be derived from the size of rotMatArray.

    Returns
    rotVecGP: rotation vector at the Gauss point.
    rotMatGP: rotation tensor at the Gauss point.
    '''

    # Memory allocations
    S = Sarray.flatten() # flattening to change dimension from (NNPEL,1) to (NNEPL,)
    rotVecGP = np.zeros(shape=3, dtype=np.float64)
    rotMatGP = np.zeros(shape=[3,3], dtype=np.float64)

    # array of rotation matrices vectorised
    rotMatVectd = np.array([rotMatArray[i].flatten() for i in range(NNPEL)])
    # interpolation of vectoried matrices
    vectdMatGP = S @ rotMatVectd # vectorised rotation matrix at Gauss point
    # devectorize rotation matrix at Gauss point
    rotMatGP = np.array([vectdMatGP[i::3] for i in range(3)]).T
    rotVecGP[:] = rotVector(rotMatGP)

    return rotVecGP, rotMatGP

In [3]:
# Test 1
# This test uses values from file 7b. For element with 2 nodes.

NNPEL = 2
rotMat = np.array([[[0.9999963089605151,0.0,0.002716995646999483],[0.0,1.0,0.0],[-0.002716995646999483,0.0,0.9999963089605151]],
    [[0.9999867473249174,0.0,0.005148317640940354],[0.0,1.0,0.0],[-0.005148317640940354,0.0,0.9999867473249174]]])

S = np.array([0.5,0.5])

_, result = piecewiseInterpRotMat(S, rotMat, NNPEL)

expResult = np.array([[0.9999915281427163, 0.0, 0.003932656643969919],
    [0.0, 1.0, 0.0],
    [-0.003932656643969919, 0.0, 0.9999915281427163]])

print('Expected result=', expResult)
print('result=', result)
# test for accuracy of calculation
if np.isclose(expResult, result).all():
    print('Result is close.')
#-------checks for properness------------#
if np.isnan(result).any():
    print('Expected result has NaN values')
else:
    print('There are no NaN values.')
# test for orthogonality of interpolated rotation matrix
if np.isclose(result.T @ result,np.eye(3)).all():
    print('Interpolated matrix is orthogonal.')
# check for properness
if np.isclose(np.linalg.det(result), 1.0):
    print('Interpolated matrix has determinant close to 1.')


Expected result= [[ 0.99999153  0.          0.00393266]
 [ 0.          1.          0.        ]
 [-0.00393266  0.          0.99999153]]
result= [[ 0.99999153  0.          0.00393266]
 [ 0.          1.          0.        ]
 [-0.00393266  0.          0.99999153]]
Result is close.
There are no NaN values.
Interpolated matrix is orthogonal.
Interpolated matrix has determinant close to 1.


Interpolation of rotation matrices representing rotation of 0.155 degree and 0.27 degree about Y axis. Interpolated vector is rotation of 0.225 degree about Y axis. 
Interpolation of small rotation works.

In [4]:
# Test 2
# This test uses values from file 7b. For element with 5 nodes.

NNPEL = 5
rotMat = np.array([[[1.0,0.0,0.0],[0.0,1.0,0.0],[0.0,0.0,1.0]],
    [[0.9999963089605151,0.0,0.002716995646999483],[0.0,1.0,0.0],[-0.002716995646999483,0.0,0.9999963089605151]],
    [[0.9999867473249174,0.0,0.005148317640940354],[0.0,1.0,0.0],[-0.005148317640940354,0.0,0.9999867473249174]],
    [[0.999973398306113,0.0,0.007294016734519014],[0.0,0.9999999999999999,0.0],[-0.007294016734519015,0.0,0.999973398306113]],
    [[0.9999580998787679,0.0,0.009154151344820142],[0.0,0.9999999999999999,0.0],[-0.009154151344820142,0.0,0.9999580998787679]]])

S, _ = interpLagGLQ(NNPEL,1)


_, result = piecewiseInterpRotMat(S, rotMat, NNPEL)

expResult = np.array([[0.9999867473249174,0.0,0.005148317640940354],
                    [0.0,1.0,0.0],
                    [-0.005148317640940354,0.0,0.9999867473249174]])

print('Expected result=', expResult)
print('result=', result)
# test for accuracy of calculation
if np.isclose(expResult, result).all():
    print('Result is close.')

#-------checks for properness------------#
if np.isnan(result).any():
    print('Expected result has NaN values')
else:
    print('There are no NaN values.')
# test for orthogonality of interpolated rotation matrix
if np.isclose(result.T @ result,np.eye(3)).all():
    print('Interpolated matrix is orthogonal.')
# check for properness
if np.isclose(np.linalg.det(result), 1.0):
    print('Interpolated matrix has determinant close to 1.')

Expected result= [[ 0.99998675  0.          0.00514832]
 [ 0.          1.          0.        ]
 [-0.00514832  0.          0.99998675]]
result= [[ 0.99998675  0.          0.00514832]
 [ 0.          1.          0.        ]
 [-0.00514832  0.          0.99998675]]
Result is close.
There are no NaN values.
Interpolated matrix is orthogonal.
Interpolated matrix has determinant close to 1.


In [5]:
# Test 3
# Check for rotation, at multiple points

NNPEL = 2
rotMat = np.array([rotTensor([0,0,0]), rotTensor([0,np.pi/2,0])])

S = np.array([[0.75,0.25],[0.5,0.5],[0.25,0.75],[0,1]])

expResult = np.array([rotTensor([0,np.deg2rad(22.5),0]), 
                    rotTensor([0,np.deg2rad(45),0]),
                    rotTensor([0,np.deg2rad(67.5),0]),
                    rotTensor([0,np.pi/2,0])])

for i in range(4):

    _, result = piecewiseInterpRotMat(S[i], rotMat, NNPEL)
    # print('Expected result=', expResult[i])
    # print('result=', result[i])
    # test for accuracy of calculation
    if np.isclose(expResult[i], result).all():
        print('Result is close.')

    #-------checks for properness------------#
    if np.isnan(result).any():
        print('Expected result has NaN values')
    else:
        print('There are no NaN values.')
    # test for orthogonality of interpolated rotation matrix
    if np.isclose(result.T @ result, np.eye(3)).all():
        print('Interpolated matrix is orthogonal.')
    else:
        print('Interpolated matrix is not orthogonal for ',i, ' iteration number.')
        print('Expected result=', expResult[i])
        print('result=', result)
    # check for properness
    if np.isclose(np.linalg.det(result), 1.0):
        print('Interpolated matrix has determinant close to 1.')
    else:
        print('Interpolated matrix does not have determinant close to 1 for ',i, ' iteration number.')

There are no NaN values.
Interpolated matrix is not orthogonal for  0  iteration number.
Expected result= [[ 0.92387953  0.          0.38268343]
 [ 0.          1.          0.        ]
 [-0.38268343  0.          0.92387953]]
result= [[ 0.75  0.    0.25]
 [ 0.    1.    0.  ]
 [-0.25  0.    0.75]]
Interpolated matrix does not have determinant close to 1 for  0  iteration number.
There are no NaN values.
Interpolated matrix is not orthogonal for  1  iteration number.
Expected result= [[ 0.70710678  0.          0.70710678]
 [ 0.          1.          0.        ]
 [-0.70710678  0.          0.70710678]]
result= [[ 0.5  0.   0.5]
 [ 0.   1.   0. ]
 [-0.5  0.   0.5]]
Interpolated matrix does not have determinant close to 1 for  1  iteration number.
There are no NaN values.
Interpolated matrix is not orthogonal for  2  iteration number.
Expected result= [[ 0.38268343  0.          0.92387953]
 [ 0.          1.          0.        ]
 [-0.92387953  0.          0.38268343]]
result= [[ 0.25  0.    0.75

Interpolation did not work for large rotation.

In [6]:
np.array([rotTensor([0,0,0]), rotTensor([0,np.pi/2,0])])

array([[[ 1.00000000e+00,  0.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00],
        [ 0.00000000e+00,  0.00000000e+00,  1.00000000e+00]],

       [[ 2.22044605e-16,  0.00000000e+00,  1.00000000e+00],
        [ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00],
        [-1.00000000e+00,  0.00000000e+00,  2.22044605e-16]]])

In [7]:
rotVector([[0.9999915281427163, 0.0, 0.003932656643969919],
    [0.0, 1.0, 0.0],
    [-0.003932656643969919, 0.0, 0.9999915281427163]])

array([0.        , 0.00393267, 0.        ])

In [8]:
np.rad2deg(0.00393267)

np.float64(0.22532539321771347)

In [9]:
rotTensor([0,np.deg2rad(22.5),0])

array([[ 0.92387953,  0.        ,  0.38268343],
       [ 0.        ,  1.        ,  0.        ],
       [-0.38268343,  0.        ,  0.92387953]])